# Journaliser, copier ou geler : trois décisions différentes

Choisir ce qu’une observation conserve du calcul et ce qui a encore le droit d’apprendre.

Nous lisons et exécutons les étapes dans l’ordre. Les cellules « Observer » montrent les valeurs et les formes ; elles ne prouvent pas la réussite des exercices.

Environnement vérifié : Python 3.11.9, PyTorch 2.10.0 CPU et TorchAudio 2.10.0 lorsque le chapitre le demande. Aucun modèle ni jeu de données n’est téléchargé pendant l’exécution.

[Revenir au chapitre](https://radiocodeur.github.io/quiz/cours.html?theme=pytorch#detacher-geler-journaliser).


## Préparer le noyau

Choisissez un environnement CPU neuf. En cas de bibliothèque manquante, installez-la dans ce noyau, puis redémarrez-le. Les sorties affichent les versions réellement utilisées.

In [1]:
import os
import sys
import tempfile

# Choisir un noyau neuf pour garantir la vérification sur CPU.
os.environ["CUDA_VISIBLE_DEVICES"] = ""
try:
    import torch
except ImportError as error:
    raise RuntimeError("PyTorch manque. Installer torch==2.10.0 dans ce noyau, puis redémarrer et relancer.") from error
torch.set_num_threads(1)
torch.manual_seed(17)
print("Python", sys.version.split()[0], "— PyTorch", torch.__version__, "— CPU")
assert not torch.cuda.is_available(), "Redémarrer dans un noyau CPU neuf."
_quiz_initial_directory = os.getcwd()
_quiz_temporary_directory = tempfile.TemporaryDirectory(prefix="quiz-progressif-")
os.chdir(_quiz_temporary_directory.name)


Python 3.11.9 — PyTorch 2.10.0+cpu — CPU


In [2]:
# Cet outil sert uniquement à afficher les états des démonstrations.
# Il ne modifie pas les tenseurs et ne fait pas partie du modèle étudié.
def observer(nom, valeur):
    if isinstance(valeur, torch.Tensor):
        contenu = valeur.detach().cpu()
        print(nom, "forme", tuple(contenu.shape), "type", contenu.dtype,
              "requires_grad", valeur.requires_grad)
        if contenu.numel() <= 16:
            print("  valeurs :", contenu.tolist())
    elif valeur is None or isinstance(valeur, (int, float, bool, str, tuple)):
        print(nom, "=", valeur)
    elif isinstance(valeur, list) and len(valeur) <= 8:
        if all(isinstance(item,(int,float,bool,str)) for item in valeur):
            print(nom, "=", valeur)
        else:
            print(nom, ":",len(valeur),"éléments ; formes des tenseurs :",
                  [tuple(item.shape) for item in valeur if isinstance(item,torch.Tensor)])
    elif isinstance(valeur,dict):
        print(nom, ": clés",list(valeur)[:12])

def observer_entrainement(espace, montrer_poids=False):
    # Une copie évite de parcourir un dictionnaire global en mutation.
    vus = set()
    for nom, objet in list(espace.items()):
        if nom.startswith('_'):
            continue
        if isinstance(objet, torch.Tensor) and objet.is_leaf and objet.requires_grad:
            parametres = [(nom, objet)]
        elif isinstance(objet, torch.nn.Module):
            parametres = [(nom + '.' + cle, parametre)
                          for cle, parametre in list(objet.named_parameters())[:4]]
        else:
            continue
        for cle, parametre in parametres:
            if id(parametre) in vus:
                continue
            vus.add(id(parametre))
            if montrer_poids:
                observer(cle, parametre)
            observer(cle + '.grad', parametre.grad)


Pour tracer une courbe, il faut une valeur de perte. Pour apprendre, il faut encore l’objet relié aux poids. Garder la mesure n’oblige donc pas à remplacer la perte.

nombre vaut 1,0 et sert au journal. valeurs vaut [1,2]. detach conserve ces nombres en coupant leur lien au graphe, mais partage leur stockage. clone fournit ensuite une copie : mettre son premier élément à 99 laisse valeurs intact.

| Opération | Tenseur ? | Ancien graphe ? | Stockage indépendant ? |
| --- | --- | --- | --- |
| perte.item() | Non : nombre Python | Non | Valeur extraite |
| valeurs.detach() | Oui | Non | Non |
| valeurs.clone() | Oui | Oui ici | Oui |
| valeurs.detach().clone() | Oui | Non | Oui |

Copier et détacher ne sont pas synonymes. Un clone direct d’un tenseur différentiable reste différentiable.

Appelez toujours backward sur perte, pas sur nombre : le float Python ne connaît plus les opérations qui l’ont produit. item convient à une mesure scalaire ; pour un tableau de sorties, gardez un tenseur détaché.

cpu() ramène les valeurs sur le processeur si elles étaient sur CUDA ; detach coupe d’abord le graphe. Ici, le tableau est déjà sur CPU. Ce transfert ne garantit pas une copie indépendante : ajoutez clone si l’archive doit ensuite être modifiée.

## Démonstration 1 · Conserver l’observation sans casser la perte

Modifier une copie et vérifier que l’apprentissage reste possible.

### Préparer les bibliothèques

Exécutez cette petite partie après la précédente. Si un objet est créé, la cellule d’observation suivante permet d’en examiner l’état.

In [3]:
import torch


### Calculer l’état suivant

Elle vaut 1 et dépend encore du poids.

In [4]:
poids = torch.tensor(1., requires_grad=True)
perte = (poids - 2.) ** 2


In [5]:
observer('poids', poids)
observer('perte', perte)

poids forme () type torch.float32 requires_grad True
  valeurs : 1.0
perte forme () type torch.float32 requires_grad True
  valeurs : 1.0


### Calculer l’état suivant

item extrait un nombre ; detach donne un tenseur hors du graphe.

In [6]:
nombre = perte.item()
valeurs = torch.stack([perte, 2 * perte])
detachees = valeurs.detach()


In [7]:
observer('nombre', nombre)
observer('valeurs', valeurs)
observer('detachees', detachees)

nombre = 1.0
valeurs forme (2,) type torch.float32 requires_grad True
  valeurs : [1.0, 2.0]
detachees forme (2,) type torch.float32 requires_grad False
  valeurs : [1.0, 2.0]


### Calculer l’état suivant

La modification concerne la copie. Le clone direct garde quant à lui la différentiabilité.

In [8]:
copie = detachees.clone()
copie[0] = 99.
clone_differentiable = valeurs.clone()


In [9]:
observer('copie', copie)
observer('clone_differentiable', clone_differentiable)

copie forme (2,) type torch.float32 requires_grad False
  valeurs : [99.0, 2.0]
clone_differentiable forme (2,) type torch.float32 requires_grad True
  valeurs : [1.0, 2.0]


### Calculer l’état suivant

cpu conserve l’archive sur le processeur ; backward utilise toujours perte.

In [10]:
archive_cpu = valeurs.detach().cpu()


In [11]:
observer('archive_cpu', archive_cpu)

archive_cpu forme (2,) type torch.float32 requires_grad False
  valeurs : [1.0, 2.0]


### Calculer les gradients

Exécutez cette petite partie après la précédente. Si un objet est créé, la cellule d’observation suivante permet d’en examiner l’état.

In [12]:
perte.backward()


In [13]:
observer_entrainement({'poids': poids, 'perte': perte, 'nombre': nombre, 'valeurs': valeurs, 'detachees': detachees, 'copie': copie, 'clone_differentiable': clone_differentiable, 'archive_cpu': archive_cpu}, montrer_poids=False)

poids.grad forme () type torch.float32 requires_grad False
  valeurs : -2.0


### Lire les sorties

Ces impressions lisent les objets calculés. Elles n’entraînent pas le modèle et ne modifient pas les tenseurs.

In [14]:
print('journal :', nombre, type(nombre).__name__)
print('original et copie :', valeurs.tolist(), copie.tolist())
print('graphe conserve par clone :', clone_differentiable.requires_grad)
print('gradient du poids :', poids.grad.item())

journal : 1.0 float
original et copie : [1.0, 2.0] [99.0, 2.0]
graphe conserve par clone : True
gradient du poids : -2.0


In [15]:
assert bool(isinstance(nombre, float) and nombre == 1), 'detacher-geler-journaliser/main: isinstance(nombre, float) and nombre == 1'
assert bool(valeurs.tolist() == [1., 2.] and copie.tolist() == [99., 2.]), 'detacher-geler-journaliser/main: valeurs.tolist() == [1., 2.] and copie.tolist() == [99., 2.]'
assert bool(detachees.data_ptr() == valeurs.data_ptr() and copie.data_ptr() != valeurs.data_ptr()), 'detacher-geler-journaliser/main: detachees.data_ptr() == valeurs.data_ptr() and copie.data_ptr() != valeurs.data_ptr()'
assert bool(clone_differentiable.requires_grad and not copie.requires_grad), 'detacher-geler-journaliser/main: clone_differentiable.requires_grad and not copie.requires_grad'
assert bool(archive_cpu.device.type == "cpu" and not archive_cpu.requires_grad), 'detacher-geler-journaliser/main: archive_cpu.device.type == "cpu" and not archive_cpu.requires_grad'
assert bool(poids.grad.item() == -2), 'detacher-geler-journaliser/main: poids.grad.item() == -2'
print("Contrôles numériques de la démonstration réussis.")

Contrôles numériques de la démonstration réussis.


L’observation et le calcul pour apprendre utilisent des objets distincts.

### Garder l’encodeur, entraîner seulement la tête

Autre besoin : réutiliser une représentation sans ajuster son encodeur. Le second programme reprend les trois mesures fictives du socle ; l’encodeur identité permet de retrouver exactement ces valeurs.

requires_grad_(False) désactive les gradients des paramètres de l’encodeur. eval agit sur le comportement de couches comme Dropout ; il ne gèle pas leurs poids. Notre couche Linear ne change pas de comportement entre ces modes. L’optimiseur ne reçoit ici que les paramètres de tete.

## Démonstration 2 · Vérifier les paramètres réellement entraînés

Un pas de classification avec un encodeur fixe.

### Préparer les bibliothèques

Exécutez cette petite partie après la précédente. Si un objet est créé, la cellule d’observation suivante permet d’en examiner l’état.

In [16]:
import torch
from torch import nn


### Calculer l’état suivant

Les valeurs sont choisies ; l’identité de l’encodeur conserve les deux descripteurs.

In [17]:
X = torch.tensor([[0.1, 1.5], [0.2, 4.5], [0.15, 2.5]])
cibles = torch.tensor([0, 1, 0], dtype=torch.long)
encodeur = nn.Linear(2, 2, bias=False)
tete = nn.Linear(2, 2)


In [18]:
observer('X', X)
observer('cibles', cibles)
observer('tete', tete)

X forme (3, 2) type torch.float32 requires_grad False
  valeurs : [[0.10000000149011612, 1.5], [0.20000000298023224, 4.5], [0.15000000596046448, 2.5]]
cibles forme (3,) type torch.int64 requires_grad False
  valeurs : [0, 1, 0]


### Appliquer le contexte explicite

Exécutez cette petite partie après la précédente. Si un objet est créé, la cellule d’observation suivante permet d’en examiner l’état.

In [19]:
with torch.no_grad():
    encodeur.weight.copy_(torch.eye(2))
    tete.weight.copy_(torch.tensor([[0., 1.], [0., -1.]]))
    tete.bias.zero_()


### Choisir les paramètres et leur mise à jour

Le gel et le mode sont explicites. L’optimiseur référence seulement la tête.

In [20]:
encodeur.requires_grad_(False)
encodeur.eval()
avant = encodeur.weight.detach().clone()
optimiseur = torch.optim.SGD(tete.parameters(), lr=0.1)


In [21]:
observer('avant', avant)

avant forme (2, 2) type torch.float32 requires_grad False
  valeurs : [[1.0, 0.0], [0.0, 1.0]]


### Construire la règle de perte

L’encodeur produit les entrées de la tête ; celle-ci construit le graphe utile à ses poids.

In [22]:
representation = encodeur(X)
scores = tete(representation)
perte = nn.CrossEntropyLoss()(scores, cibles)


In [23]:
observer('representation', representation)
observer('scores', scores)
observer('perte', perte)

representation forme (3, 2) type torch.float32 requires_grad False
  valeurs : [[0.10000000149011612, 1.5], [0.20000000298023224, 4.5], [0.15000000596046448, 2.5]]
scores forme (3, 2) type torch.float32 requires_grad True
  valeurs : [[1.5, -1.5], [4.5, -4.5], [2.5, -2.5]]
perte forme () type torch.float32 requires_grad True
  valeurs : 3.0184755325317383


### Calculer les gradients

La tête a des gradients ; l’encodeur reste identique.

In [24]:
perte.backward()


In [25]:
observer_entrainement({'X': X, 'cibles': cibles, 'encodeur': encodeur, 'tete': tete, 'avant': avant, 'optimiseur': optimiseur, 'representation': representation, 'scores': scores, 'perte': perte}, montrer_poids=False)

encodeur.weight.grad = None
tete.weight.grad forme (2, 2) type torch.float32 requires_grad False
  valeurs : [[0.06474293768405914, 1.470524787902832], [-0.06474293768405914, -1.4705246686935425]]
tete.bias.grad forme (2,) type torch.float32 requires_grad False
  valeurs : [0.3152526617050171, -0.3152526617050171]


### Appliquer une mise à jour

Exécutez cette petite partie après la précédente. Si un objet est créé, la cellule d’observation suivante permet d’en examiner l’état.

In [26]:
optimiseur.step()


In [27]:
observer_entrainement({'X': X, 'cibles': cibles, 'encodeur': encodeur, 'tete': tete, 'avant': avant, 'optimiseur': optimiseur, 'representation': representation, 'scores': scores, 'perte': perte}, montrer_poids=True)

encodeur.weight forme (2, 2) type torch.float32 requires_grad False
  valeurs : [[1.0, 0.0], [0.0, 1.0]]
encodeur.weight.grad = None
tete.weight forme (2, 2) type torch.float32 requires_grad True
  valeurs : [[-0.006474293768405914, 0.8529475331306458], [0.006474293768405914, -0.8529475331306458]]
tete.weight.grad forme (2, 2) type torch.float32 requires_grad False
  valeurs : [[0.06474293768405914, 1.470524787902832], [-0.06474293768405914, -1.4705246686935425]]
tete.bias forme (2,) type torch.float32 requires_grad True
  valeurs : [-0.03152526542544365, 0.03152526542544365]
tete.bias.grad forme (2,) type torch.float32 requires_grad False
  valeurs : [0.3152526617050171, -0.3152526617050171]


### Lire les sorties

Ces impressions lisent les objets calculés. Elles n’entraînent pas le modèle et ne modifient pas les tenseurs.

In [28]:
print('gradient encodeur :', encodeur.weight.grad)
print('gradient tete present :', tete.weight.grad is not None)
print('encodeur identique :', torch.equal(avant, encodeur.weight))

gradient encodeur : None
gradient tete present : True
encodeur identique : True


In [29]:
assert bool(encodeur.weight.grad is None and tete.weight.grad is not None), 'detacher-geler-journaliser/gel: encodeur.weight.grad is None and tete.weight.grad is not None'
assert bool(torch.equal(avant, encodeur.weight)), 'detacher-geler-journaliser/gel: torch.equal(avant, encodeur.weight)'
assert bool(torch.equal(representation, X) and not representation.requires_grad), 'detacher-geler-journaliser/gel: torch.equal(representation, X) and not representation.requires_grad'
print("Contrôles numériques de la démonstration réussis.")

Contrôles numériques de la démonstration réussis.


Une partie gelée peut alimenter une autre partie qui continue à apprendre.

## À vous de modifier le calcul

Les deux cellules suivantes sont prêtes à éditer. Si vous les exécutez sans les compléter, elles le signalent et ne prétendent pas que vous avez réussi. Une réponse fausse déclenche le contrôle correspondant.

### Exercice 1

Nous modifions la première case d’un tenseur détaché. Que vaut la première case de la source ?

In [30]:
exo_x = torch.tensor([1.,2.],requires_grad=True)
exo_detache = exo_x.detach()
exo_detache[0] = 9.
exo_source = ...  # Remplacer les points par votre réponse.
if exo_source is Ellipsis or exo_source is None:
    print("À compléter : cet exercice n’est pas encore réalisé.")
else:
    assert abs(float(exo_source)-exo_x[0].item()) < 1e-6
    print("Contrôle réussi pour la réponse fournie. Expliquez maintenant pourquoi.")


À compléter : cet exercice n’est pas encore réalisé.


<details>
<summary>Indice pour l’exercice 1</summary>

detach retire le lien au graphe, mais partage encore le stockage.

</details>

### Exercice 2

Construisez cette fois une copie détachée indépendante avant de modifier sa première case.

In [31]:
exo_x = torch.tensor([1.,2.],requires_grad=True)
exo_copie = ...  # Remplacer les points par votre réponse.
if exo_copie is Ellipsis or exo_copie is None:
    print("À compléter : cet exercice n’est pas encore réalisé.")
else:
    exo_copie[0]=9.
    assert exo_x[0].item() == 1.
    assert not exo_copie.requires_grad
    print("Contrôle réussi pour la réponse fournie. Expliquez maintenant pourquoi.")


À compléter : cet exercice n’est pas encore réalisé.


<details>
<summary>Indice pour l’exercice 2</summary>

Deux propriétés sont demandées : aucun lien au graphe et aucun stockage partagé.

</details>

## Comparer avec les solutions

Ouvrez-les après avoir formulé votre réponse et votre explication. Les solutions sont également exécutées par le vérificateur du dépôt.

<details>
<summary>Solution de l’exercice 1</summary>

```python
exo_x = torch.tensor([1.,2.],requires_grad=True)
exo_detache = exo_x.detach()
exo_detache[0] = 9.
exo_source = 9.
if exo_source is Ellipsis or exo_source is None:
    print("À compléter : cet exercice n’est pas encore réalisé.")
else:
    assert abs(float(exo_source)-exo_x[0].item()) < 1e-6
    print("Contrôle réussi pour la réponse fournie. Expliquez maintenant pourquoi.")
```

</details>

<details>
<summary>Solution de l’exercice 2</summary>

```python
exo_x = torch.tensor([1.,2.],requires_grad=True)
exo_copie = exo_x.detach().clone()
if exo_copie is Ellipsis or exo_copie is None:
    print("À compléter : cet exercice n’est pas encore réalisé.")
else:
    exo_copie[0]=9.
    assert exo_x[0].item() == 1.
    assert not exo_copie.requires_grad
    print("Contrôle réussi pour la réponse fournie. Expliquez maintenant pourquoi.")
```

</details>

In [34]:
os.chdir(_quiz_initial_directory)
_quiz_temporary_directory.cleanup()